# GNN inputs: node features and node2vec embeddings (honest vs. cheating graphs)

Turns the safe / leaky / null graphs from notebook 07 into `torch_geometric` `Data` objects for GCN, GraphSAGE and GAT, and works out which node features carry signal. **No train/test split is made here**; each `Data` carries `patient_idx` so grouped splits can be added later.

Edges and features can each leak, so we build a 2x2 grid plus a null-graph control:

| | features: safe | features: leaky |
|---|---|---|
| **graph: safe** | honest | leaky features only |
| **graph: leaky** | leaky edges only | fully leaky |
| **graph: null** | control | |

No feature is computed from outcomes. Labels appear only in the descriptive importance checks in section 4.

In [ ]:
# --- paste the setup cell from notebook 07 here (defines `p`, TASK, plotting setup) ---
# --- and the same imports for task_graphs_from_labels and analyze_task -------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from ehrgraph.gnn.node_features import (
    FEATURE_SETS, LEAKABLE_BLOCKS, make_blocks, build_pyg_data, check_data,
    block_ablation, column_auroc, code_gap_table, top_codes, ablation_rows,
)

## 1. Graphs and analysis (reuse from notebook 07 if still in memory)

In [ ]:
CLMBR_PATH = None   # full release only: Path(ASSETS) / "features" / "clmbr_features.pkl"
                    # with None the CLMBR block is simply skipped

labels = p.labels(TASK)
nets = task_graphs_from_labels(p.diagnoses, labels, TASK, clmbr_path=CLMBR_PATH)
results = analyze_task(nets)      # supplies centralities and communities; skip if already in memory
y = np.array([nets.labels[n] for n in nets.nodes])
groups = pd.factorize(pd.Series([n[0] for n in nets.nodes]))[0]     # patient id, for grouped CV in section 4
print(f"{len(nets.nodes):,} stays, {nets.prevalence:.1%} with the outcome, "
      f"CLMBR on {len(nets.feature_nodes):,} stays")

## 2. Feature blocks

Each block is a named group of columns, built per grid cell. The cell decides which graph supplies the graph-derived blocks and which diagnosis vectors (safe / leaky) feed the diagnosis blocks.

| block | what it is | lecture category | leak path |
|---|---|---|---|
| `dx_svd` | TF-IDF of the diagnosis vector, then 64-dim SVD | feature encoder, no graph | via the source |
| `dx_summary` | log number of distinct diagnoses (cheating profiles have ~2x) | feature | via the source |
| `structure` | log degree, clustering, k-core, same-patient neighbours, centrality percentiles | hand-engineered graph features | via the graph |
| `rwpe` | random-walk return probabilities, computed exactly | hand-engineered, captures structural *role* | via the graph |
| `n2v` | **node2vec**: biased walks, then skip-gram with negative sampling | learned shallow embedding (lookup table) | via the graph |
| `clmbr` | Stanford's CLMBR patient embedding at prediction time | pretrained deep encoder on the record, no graph | none (verify, see notes) |
| `time` | prediction year | ablation only | none |

node2vec settings (`N2V`): 64 dims, 10 walks of length 30 per node, window 5, 5 negative samples, p = q = 1 (unbiased, i.e. DeepWalk). Treat `p` and `q` as hyperparameters to sweep later.

In [ ]:
GRID = [("safe", "safe"), ("safe", "leaky"), ("leaky", "safe"), ("leaky", "leaky"), ("null", "safe")]
N2V = dict(dim=64, num_walks=10, walk_length=30, window=5, negatives=5, epochs=5, p=1.0, q=1.0)

n2v_cache = {}          # node2vec depends on the graph only: train it once per graph (~1 min each)
blocks = {
    (g, s): make_blocks(nets, g, s, centrality=results.centrality[g],
                        svd_dim=64, rw_steps=8, n2v=N2V, n2v_cache=n2v_cache)
    for g, s in GRID
}

b = blocks[("safe", "safe")]
print({k: v.values.shape[1] for k, v in b.items()})
print("isolated nodes (no walk possible, zero n2v vector):",
      {g: f"{blk.meta['isolated']:,} of {len(nets.nodes):,}" for g, blk in n2v_cache.items()})

In [ ]:
# what an SVD axis means, in diagnosis codes (first three components, honest vectors)
for c in range(3):
    print(f"component {c}"); display(top_codes(blocks[("safe", "safe")]["dx_svd"], c, n=8))

### Sanity check: does node2vec reflect the graph?

Cosine similarity of node2vec vectors for pairs in the same Leiden community vs. random pairs (connected nodes only). The safe and leaky graphs should show a clear gap; the null graph, which has no community structure beyond its degrees, should show almost none. Embeddings are only comparable *within* one graph, so read each condition separately.

In [ ]:
rng = np.random.default_rng(0)
for g in ("safe", "leaky", "null"):
    z = n2v_cache[g].values[:, :-1]
    ok = n2v_cache[g].values[:, -1] == 0
    part = results.communities[g]["leiden"].to_dict()
    comm = np.array([part.get(n, -1) for n in nets.nodes])
    zn = z / np.clip(np.linalg.norm(z, axis=1, keepdims=True), 1e-9, None)
    idx = np.flatnonzero(ok & (comm >= 0))
    a, c = rng.choice(idx, 20000), rng.choice(idx, 20000)
    same = (comm[a] == comm[c]) & (a != c)
    cos = (zn[a] * zn[c]).sum(1)
    print(f"{g:6s} same community {cos[same].mean():.2f}   different {cos[~same].mean():.2f}   ({same.sum():,} same-community pairs)")

## 3. CLMBR next to the graphs

The CLMBR embedding is the honest reference, so the first question is how it relates to each graph. `dirichlet_energy_clmbr` (from notebook 07's analysis) measures how different CLMBR vectors are across the graph's edges: lower means neighbours look more alike to CLMBR. If the honest graph's energy sits well below the null's, the diagnosis graph and CLMBR agree about what "similar patient" means.

In [ ]:
clmbr_rows = pd.DataFrame([r for r in results.rows if r["metric"] in ("dirichlet_energy_clmbr", "clmbr_coverage")])
if clmbr_rows.empty:
    print("No CLMBR in this run (CLMBR_PATH is None).")
else:
    display(clmbr_rows.pivot_table(index="metric", columns="condition", values="value"))

## 4. Which features matter

Three views, from finest to coarsest. These are **descriptive** (like the paper's homophily, they use all outcomes) and use no GNN.

1. **Diagnosis codes**: share of positive vs. negative nodes that have each code. This is where a leak shows up by name.
2. **Single columns**: AUROC of each hand-engineered structure column.
3. **Blocks**: gradient-boosted trees on each block alone, and on everything except that block. AUROC is out-of-fold with folds grouped by patient, so a patient's other stays never sit in the evaluation fold. This internal grouping is for the diagnostic only and is not a GNN split.

A single leaky code can be diluted inside a 64-dim SVD, so view 1 is the direct leak detector; view 3 shows what survives into the embedding.

In [ ]:
for s in ("safe", "leaky"):
    matrix, codes = nets.vectors[s]
    print(f"--- top code gaps, {s} vectors")
    display(code_gap_table(matrix, codes, y, top=10))

In [ ]:
display(column_auroc(blocks[("safe", "safe")]["structure"], y))

In [ ]:
NAMES = ["dx_svd", "dx_summary", "structure", "rwpe", "n2v", "time", "clmbr"]   # missing blocks are skipped
ablation = {(g, s): block_ablation(blocks[(g, s)], NAMES, y, groups) for g, s in GRID}
cols = {f"graph={g}, feat={s}": t.set_index("block") for (g, s), t in ablation.items()}
tbl = pd.concat({k: v["alone_auroc"] for k, v in cols.items()}, axis=1)
display(tbl.round(3))
display(pd.concat({k: v["auroc_lost_when_dropped"] for k, v in cols.items()}, axis=1).round(3))

In [ ]:
ax = tbl.drop(index="ALL").plot.bar(figsize=(9, 3.5), rot=0)
ax.axhline(0.5, color="grey", lw=0.8, ls=":"); ax.set_ylabel("out-of-fold AUROC, block alone")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()

In [ ]:
rows = []
for (g, s), t in ablation.items():
    rows += ablation_rows(t, TASK, g, s, "omop_concept_binary_perpatient_tfidf_mutual_k10")
# write_aggregate(rows, p.results / f"node_feature_importance_{TASK}.csv")   # check it accepts these condition names

## 5. Build the PyG datasets

Feature sets without diagnosis blocks (`structure`, `n2v`, `clmbr`, `clmbr_n2v`) ignore the feature source, so only `source="safe"` is built for them. `x` is **unscaled**; call `standardize(data, train_mask)` once splits exist.

In [ ]:
datasets = {}
for g, s in GRID:
    for fs, names in FEATURE_SETS.items():
        if not any(n in LEAKABLE_BLOCKS for n in names) and s == "leaky":
            continue                      # identical to the s == "safe" cell
        if not all(n in blocks[(g, s)] for n in names):
            continue                      # e.g. clmbr without the full release
        datasets[(g, s, fs)] = build_pyg_data(nets, g, s, fs, blocks=blocks[(g, s)])

for d in datasets.values():
    check_data(d)

summary = pd.DataFrame(
    [{"graph": g, "features": s, "feature_set": fs, "x_dim": d.x.shape[1],
      "undirected_edges": d.edge_index.shape[1] // 2,
      "same_patient_edge_share": float(d.edge_attr.mean())}
     for (g, s, fs), d in datasets.items()]
)
display(summary.round(3))

In [ ]:
# Optional sensitivity: the hypertension paper found same-patient edges carry much of the signal.
no_same = build_pyg_data(nets, "safe", "safe", "dx_n2v", blocks=blocks[("safe", "safe")],
                         drop_same_patient_edges=True)
print(no_same.edge_index.shape[1] // 2, "edges without same-patient links")
# note: n2v was trained WITH those edges; for a clean version re-run make_blocks on a graph without them.

## 6. Save

These files hold patient-level data (anonymised node order, no raw patient IDs, but still sensitive). Keep them out of git. Loading needs `weights_only=False` because they contain PyG `Data` objects.

In [ ]:
out = p.results / f"gnn_inputs_{TASK}.pt"
torch.save(datasets, out)
# datasets = torch.load(out, weights_only=False)
print(out, f"{out.stat().st_size / 1e6:.1f} MB")

## Later (not here)

- Patient-grouped train/val/test splits using `data.patient_idx`, then `standardize(data, train_mask)`.
- Baselines on the same inputs: logistic regression on `clmbr` (the EHRSHOT-style probe), and on `n2v` alone (the lecture's embedding + classifier pipeline), with no GNN.
- GCN / GraphSAGE / GAT on `datasets[(graph, source, feature_set)]`; an MLP on the same `x` as the no-graph control. Only `GCNConv` uses `edge_weight`.
- A `p`, `q` sweep for node2vec, and several node2vec seeds (embeddings change with the seed).